In [1]:
import numpy as np

def softmax(x, axis=-1):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / e_x.sum(axis=axis, keepdims=True)

def self_attention(X, W_Q, W_K, W_V):
    """
    Compute scaled dot-product self-attention.

    Args:
        X   : (seq_len, d_model)  — input embeddings
        W_Q : (d_model, d_k)      — query weight matrix
        W_K : (d_model, d_k)      — key weight matrix
        W_V : (d_model, d_v)      — value weight matrix

    Returns:
        output        : (seq_len, d_v) — attended output
        attn_weights  : (seq_len, seq_len) — attention score matrix
    """
    d_k = W_K.shape[1]

    Q = X @ W_Q 
    K = X @ W_K  
    V = X @ W_V

    scores = (Q @ K.T) / np.sqrt(d_k) 
    attn_weights = softmax(scores, axis=-1)
    output = attn_weights @ V    

    return output, attn_weights

In [3]:
if __name__ == "__main__":
    np.random.seed(42)

    tokens = ["What", "are", "the", "symptoms", "of", "diabetes", "?"]
    seq_len = len(tokens) 
    d_model = 8             
    d_k = 4             
    d_v = 4           
    X = np.random.randn(seq_len, d_model)

    W_Q = np.random.randn(d_model, d_k)
    W_K = np.random.randn(d_model, d_k)
    W_V = np.random.randn(d_model, d_v)

    output, attn_weights = self_attention(X, W_Q, W_K, W_V)

    print("Tokens:", tokens)
    print("\nAttention weight matrix (rows = query token, cols = key token):")
    print(np.array2string(attn_weights, precision=3, suppress_small=True))

    print("\nFor token 'symptoms' (index 3), attention over all tokens:")
    for tok, w in zip(tokens, attn_weights[3]):
        bar = "█" * int(w * 30)
        print(f"  {tok:12s}  {w:.3f}  {bar}")

    print("\nEncoder output shape:", output.shape)

Tokens: ['What', 'are', 'the', 'symptoms', 'of', 'diabetes', '?']

Attention weight matrix (rows = query token, cols = key token):
[[0.    0.    0.    0.    1.    0.    0.   ]
 [0.001 0.    0.    0.    0.    0.002 0.997]
 [0.916 0.    0.047 0.    0.    0.    0.037]
 [0.    0.639 0.    0.335 0.025 0.001 0.   ]
 [0.    0.    0.    0.    0.    0.    1.   ]
 [0.    0.306 0.    0.04  0.654 0.    0.   ]
 [0.    0.006 0.    0.    0.994 0.    0.   ]]

For token 'symptoms' (index 3), attention over all tokens:
  What          0.000  
  are           0.639  ███████████████████
  the           0.000  
  symptoms      0.335  ██████████
  of            0.025  
  diabetes      0.001  
  ?             0.000  

Encoder output shape: (7, 4)
